In [ ]:
!pip -q install wandb

In [1]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA not available")

torch: 2.11.0+cu128
cuda available: True
cuda: 12.8
gpu: NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
import os

os.environ["WANDB_START_METHOD"] = "thread"
os.environ["WANDB__SERVICE_WAIT"] = "300"
import math
import random
import numpy as np
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

import wandb
from PIL import Image

# 재현성
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# WandB 로그인
wandb.login()

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\dochi\_netrc.


device: cuda


wandb: Currently logged in as: skqorekdls (skqorekdls-kwangwoon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
BASELINE_CONFIG = {
    "project_name": "mlops-cifar100",
    "run_name": "baseline_resnet18_adamw_lr1e-3_bs128",
    "dataset": "CIFAR100",
    "num_classes": 100,
    "image_size": 224,
    "batch_size": 128,
    "epochs": 15,              
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "optimizer": "AdamW",      
    "scheduler": "cosine",     
    "num_workers": 0,
    "pin_memory": torch.cuda.is_available(),
    "val_ratio": 0.1,
    "model_name": "resnet18",
    "use_pretrained": True,
    "augmentation": "basic",
    "label_smoothing": 0.0,
    "log_pred_every": 5,       
    "max_pred_images": 12,
    "pin_memory": True,
}

In [10]:
from torchvision import transforms

def get_transforms(image_size=224, augmentation="basic"):
    if augmentation == "basic":
        train_tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomCrop(image_size, padding=8),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5071, 0.4867, 0.4408],
                                 std=[0.2675, 0.2565, 0.2761]),
        ])

    elif augmentation == "strong":
        train_tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomCrop(image_size, padding=8),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5071, 0.4867, 0.4408],
                                 std=[0.2675, 0.2565, 0.2761]),
        ])

    elif augmentation == "light":
        train_tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5071, 0.4867, 0.4408],
                                 std=[0.2675, 0.2565, 0.2761]),
        ])

    else:
        raise ValueError(f"Unsupported augmentation: {augmentation}")

    eval_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071, 0.4867, 0.4408],
                             std=[0.2675, 0.2565, 0.2761]),
    ])
    return train_tf, eval_tf


class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform is not None:
            x = self.transform(x)
        return x, y


def build_dataloaders(config):
    train_tf, eval_tf = get_transforms(
    image_size=config["image_size"],
    augmentation=config.get("augmentation", "basic")
    )

    full_train_raw = datasets.CIFAR100(
        root="./data", train=True, download=True, transform=None
    )
    test_set = datasets.CIFAR100(
        root="./data", train=False, download=True, transform=eval_tf
    )

    class_names = full_train_raw.classes

    val_size = int(len(full_train_raw) * config["val_ratio"])
    train_size = len(full_train_raw) - val_size

    generator = torch.Generator().manual_seed(42)
    train_subset_raw, val_subset_raw = random_split(
        full_train_raw, [train_size, val_size], generator=generator
    )

    train_set = TransformSubset(train_subset_raw, transform=train_tf)
    val_set = TransformSubset(val_subset_raw, transform=eval_tf)

    train_loader = DataLoader(
        train_set,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"],
        pin_memory=config["pin_memory"],
    )
    val_loader = DataLoader(
        val_set,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=config["pin_memory"],
    )
    test_loader = DataLoader(
        test_set,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=config["pin_memory"],
    )

    return train_loader, val_loader, test_loader, class_names

In [11]:
def build_model(config):
    model_name = config["model_name"]
    use_pretrained = config["use_pretrained"]
    num_classes = config["num_classes"]

    if model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "resnet34":
        weights = models.ResNet34_Weights.DEFAULT if use_pretrained else None
        model = models.resnet34(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if use_pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    return model.to(device)

def build_optimizer(config, model):
    if config["optimizer"] == "AdamW":
        optimizer = optim.AdamW(
            model.parameters(),
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"],
        )
    elif config["optimizer"] == "SGD":
        optimizer = optim.SGD(
            model.parameters(),
            lr=config["learning_rate"],
            momentum=0.9,
            weight_decay=config["weight_decay"],
            nesterov=True,
        )
    else:
        raise ValueError(f"Unsupported optimizer: {config['optimizer']}")
    return optimizer


def build_scheduler(config, optimizer, total_epochs):
    if config["scheduler"] == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)
    elif config["scheduler"] == "none":
        return None
    else:
        raise ValueError(f"Unsupported scheduler: {config['scheduler']}")

In [12]:
def accuracy_from_logits(logits, labels):
    preds = logits.argmax(dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct / total


def denormalize(img_tensor):
    mean = torch.tensor([0.5071, 0.4867, 0.4408], device=img_tensor.device).view(3, 1, 1)
    std = torch.tensor([0.2675, 0.2565, 0.2761], device=img_tensor.device).view(3, 1, 1)
    img = img_tensor * std + mean
    img = img.clamp(0, 1)
    return img


@torch.no_grad()
def log_prediction_examples(run, model, loader, class_names, epoch, max_images=12):
    model.eval()
    images, labels = next(iter(loader))
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)
    confs = probs.max(dim=1).values

    logged_images = []
    max_n = min(max_images, images.size(0))

    for i in range(max_n):
        img = denormalize(images[i]).detach().cpu()
        caption = (
            f"GT: {class_names[labels[i].item()]}\n"
            f"Pred: {class_names[preds[i].item()]}\n"
            f"Conf: {confs[i].item():.4f}"
        )
        logged_images.append(wandb.Image(img, caption=caption))

    run.log({"pred_examples": logged_images, "epoch": epoch})

In [13]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        running_correct += (logits.argmax(dim=1) == labels).sum().item()
        running_total += labels.size(0)

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * labels.size(0)
        running_correct += (logits.argmax(dim=1) == labels).sum().item()
        running_total += labels.size(0)

        all_preds.extend(logits.argmax(dim=1).detach().cpu().tolist())
        all_labels.extend(labels.detach().cpu().tolist())

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc, all_preds, all_labels

In [14]:
def run_experiment(config):
    seed_everything(42)

    train_loader, val_loader, test_loader, class_names = build_dataloaders(config)

    with wandb.init(
        project=config["project_name"],
        name=config["run_name"],
        config=config,
    ) as run:
        cfg = wandb.config

        model = build_model(cfg)
        criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
        optimizer = build_optimizer(cfg, model)
        scheduler = build_scheduler(cfg, optimizer, cfg["epochs"])

        # 원하면 gradient/parameter 히스토그램도 볼 수 있음
        # run.watch(model, log="all", log_freq=200)

        best_val_acc = 0.0
        best_state = None
        best_epoch = -1

        for epoch in range(1, cfg["epochs"] + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
            val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader, criterion)

            if scheduler is not None:
                scheduler.step()

            current_lr = optimizer.param_groups[0]["lr"]

            run.log({
                "epoch": epoch,
                "train/loss": train_loss,
                "train/acc": train_acc,
                "val/loss": val_loss,
                "val/acc": val_acc,
                "lr": current_lr,
            })

            print(
                f"[{epoch:02d}/{cfg['epochs']}] "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} lr={current_lr:.6f}"
            )

            if epoch % cfg["log_pred_every"] == 0 or epoch == 1 or epoch == cfg["epochs"]:
                log_prediction_examples(
                    run=run,
                    model=model,
                    loader=val_loader,
                    class_names=class_names,
                    epoch=epoch,
                    max_images=cfg["max_pred_images"],
                )

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch
                best_state = deepcopy(model.state_dict())

        # best model 복원
        model.load_state_dict(best_state)

        # final validation confusion matrix
        val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader, criterion)
        run.log({
            "val/confusion_matrix": wandb.plot.confusion_matrix(
                preds=val_preds,
                y_true=val_labels,
                class_names=class_names
            )
        })

        # test 평가
        test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
        run.log({
            "test/loss": test_loss,
            "test/acc": test_acc,
            "test/confusion_matrix": wandb.plot.confusion_matrix(
                preds=test_preds,
                y_true=test_labels,
                class_names=class_names
            )
        })

        run.summary["best_val_acc"] = best_val_acc
        run.summary["best_epoch"] = best_epoch
        run.summary["final_test_acc"] = test_acc
        run.summary["final_test_loss"] = test_loss

        # best model 저장
        os.makedirs("checkpoints", exist_ok=True)
        model_path = f"checkpoints/{cfg['run_name']}_best.pt"
        torch.save(best_state, model_path)
        run.save(model_path)

        print("\n===== Final Result =====")
        print(f"best_val_acc : {best_val_acc:.4f} (epoch {best_epoch})")
        print(f"test_loss    : {test_loss:.4f}")
        print(f"test_acc     : {test_acc:.4f}")

        return {
            "best_val_acc": best_val_acc,
            "best_epoch": best_epoch,
            "test_loss": test_loss,
            "test_acc": test_acc,
        }

In [15]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
#Baseline
results = run_experiment(BASELINE_CONFIG)
results

100%|██████████| 169M/169M [00:05<00:00, 29.4MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 192MB/s]


[01/15] train_loss=1.8327 train_acc=0.4993 val_loss=1.6070 val_acc=0.5398 lr=0.000989
[02/15] train_loss=1.1217 train_acc=0.6722 val_loss=1.2202 val_acc=0.6474 lr=0.000957
[03/15] train_loss=0.8655 train_acc=0.7386 val_loss=1.1601 val_acc=0.6650 lr=0.000905
[04/15] train_loss=0.6774 train_acc=0.7912 val_loss=1.0374 val_acc=0.7090 lr=0.000835
[05/15] train_loss=0.5191 train_acc=0.8376 val_loss=0.9814 val_acc=0.7266 lr=0.000750
[06/15] train_loss=0.3763 train_acc=0.8819 val_loss=0.9989 val_acc=0.7300 lr=0.000655
[07/15] train_loss=0.2600 train_acc=0.9179 val_loss=0.9706 val_acc=0.7364 lr=0.000552
[08/15] train_loss=0.1800 train_acc=0.9443 val_loss=0.9170 val_acc=0.7644 lr=0.000448
[09/15] train_loss=0.1072 train_acc=0.9691 val_loss=0.8693 val_acc=0.7774 lr=0.000345
[10/15] train_loss=0.0628 train_acc=0.9837 val_loss=0.8442 val_acc=0.7908 lr=0.000250
[11/15] train_loss=0.0361 train_acc=0.9918 val_loss=0.8126 val_acc=0.7994 lr=0.000165
[12/15] train_loss=0.0232 train_acc=0.9958 val_loss=0.

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



===== Final Result =====
best_val_acc : 0.8060 (epoch 12)
test_loss    : 0.8380
test_acc     : 0.7961


epoch,▁▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇██
lr,██▇▇▆▆▅▄▃▃▂▂▁▁▁
test/acc,▁
test/loss,▁
train/acc,▁▃▄▅▆▆▇▇███████
train/loss,█▅▄▄▃▂▂▂▁▁▁▁▁▁▁
val/acc,▁▄▄▅▆▆▆▇▇██████
val/loss,█▅▄▃▃▃▃▂▂▁▁▁▁▁▁
best_epoch,12
best_val_acc,0.806
epoch,15


{'best_val_acc': 0.806,
 'best_epoch': 12,
 'test_loss': 0.8380005078792572,
 'test_acc': 0.7961}

In [ ]:
# Learning rate 변화
config_lr = dict(BASELINE_CONFIG)
config_lr["run_name"] = "resnet18_adamw_lr3e-4_bs128"
config_lr["learning_rate"] = 3e-4

run_experiment(config_lr)

100%|██████████| 169M/169M [00:05<00:00, 30.7MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


[01/15] train_loss=1.5584 train_acc=0.5993 val_loss=1.0821 val_acc=0.6856 lr=0.000297
[02/15] train_loss=0.7709 train_acc=0.7743 val_loss=0.8643 val_acc=0.7420 lr=0.000287
[03/15] train_loss=0.5466 train_acc=0.8361 val_loss=0.7913 val_acc=0.7578 lr=0.000271
[04/15] train_loss=0.3910 train_acc=0.8825 val_loss=0.7952 val_acc=0.7730 lr=0.000250
[05/15] train_loss=0.2681 train_acc=0.9217 val_loss=0.7600 val_acc=0.7826 lr=0.000225
[06/15] train_loss=0.1769 train_acc=0.9498 val_loss=0.7791 val_acc=0.7862 lr=0.000196
[07/15] train_loss=0.1219 train_acc=0.9674 val_loss=0.7234 val_acc=0.7938 lr=0.000166
[08/15] train_loss=0.0742 train_acc=0.9820 val_loss=0.6891 val_acc=0.8092 lr=0.000134
[09/15] train_loss=0.0442 train_acc=0.9911 val_loss=0.6895 val_acc=0.8154 lr=0.000104
[10/15] train_loss=0.0259 train_acc=0.9958 val_loss=0.6681 val_acc=0.8222 lr=0.000075
[11/15] train_loss=0.0171 train_acc=0.9976 val_loss=0.6517 val_acc=0.8286 lr=0.000050
[12/15] train_loss=0.0124 train_acc=0.9985 val_loss=0.

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



===== Final Result =====
best_val_acc : 0.8376 (epoch 14)
test_loss    : 0.6733
test_acc     : 0.8310


epoch,▁▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇██
lr,██▇▇▆▆▅▄▃▃▂▂▁▁▁
test/acc,▁
test/loss,▁
train/acc,▁▄▅▆▇▇▇████████
train/loss,█▄▃▃▂▂▂▁▁▁▁▁▁▁▁
val/acc,▁▄▄▅▅▆▆▇▇▇█████
val/loss,█▅▃▃▃▃▂▂▂▁▁▁▁▁▁
best_epoch,14
best_val_acc,0.8376
epoch,15


{'best_val_acc': 0.8376,
 'best_epoch': 14,
 'test_loss': 0.6733042321205139,
 'test_acc': 0.831}

In [ ]:
#Optimizer 변화
config_sgd = dict(BASELINE_CONFIG)
config_sgd["run_name"] = "resnet18_sgd_lr1e-2_bs128"
config_sgd["optimizer"] = "SGD"
config_sgd["learning_rate"] = 1e-2

run_experiment(config_sgd)

100%|██████████| 169M/169M [00:06<00:00, 25.1MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


[01/15] train_loss=1.5891 train_acc=0.5888 val_loss=0.9389 val_acc=0.7164 lr=0.009891
[02/15] train_loss=0.7275 train_acc=0.7795 val_loss=0.7836 val_acc=0.7578 lr=0.009568
[03/15] train_loss=0.5309 train_acc=0.8364 val_loss=0.7234 val_acc=0.7794 lr=0.009045
[04/15] train_loss=0.3914 train_acc=0.8804 val_loss=0.6838 val_acc=0.7906 lr=0.008346
[05/15] train_loss=0.2818 train_acc=0.9160 val_loss=0.6713 val_acc=0.8028 lr=0.007500
[06/15] train_loss=0.1946 train_acc=0.9458 val_loss=0.6614 val_acc=0.8112 lr=0.006545
[07/15] train_loss=0.1349 train_acc=0.9655 val_loss=0.6445 val_acc=0.8140 lr=0.005523
[08/15] train_loss=0.0947 train_acc=0.9782 val_loss=0.6214 val_acc=0.8262 lr=0.004477
[09/15] train_loss=0.0643 train_acc=0.9890 val_loss=0.6048 val_acc=0.8326 lr=0.003455
[10/15] train_loss=0.0473 train_acc=0.9932 val_loss=0.6029 val_acc=0.8310 lr=0.002500
[11/15] train_loss=0.0374 train_acc=0.9958 val_loss=0.5976 val_acc=0.8334 lr=0.001654
[12/15] train_loss=0.0316 train_acc=0.9970 val_loss=0.

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



===== Final Result =====
best_val_acc : 0.8376 (epoch 14)
test_loss    : 0.6320
test_acc     : 0.8292


epoch,▁▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇██
lr,██▇▇▆▆▅▄▃▃▂▂▁▁▁
test/acc,▁
test/loss,▁
train/acc,▁▄▅▆▇▇▇████████
train/loss,█▄▃▃▂▂▁▁▁▁▁▁▁▁▁
val/acc,▁▃▅▅▆▆▇▇███████
val/loss,█▅▄▃▃▂▂▂▁▁▁▁▁▁▁
best_epoch,14
best_val_acc,0.8376
epoch,15


{'best_val_acc': 0.8376,
 'best_epoch': 14,
 'test_loss': 0.6320365148067474,
 'test_acc': 0.8292}

In [ ]:
#batch_size 변화
config_bs = dict(BASELINE_CONFIG)
config_bs["run_name"] = "resnet18_adamw_lr1e-3_bs64"
config_bs["batch_size"] = 64

run_experiment(config_bs)

[01/15] train_loss=2.0710 train_acc=0.4417 val_loss=1.8394 val_acc=0.4970 lr=0.000989
[02/15] train_loss=1.3132 train_acc=0.6204 val_loss=1.4138 val_acc=0.6060 lr=0.000957
[03/15] train_loss=1.0294 train_acc=0.6935 val_loss=1.2873 val_acc=0.6454 lr=0.000905
[04/15] train_loss=0.8331 train_acc=0.7482 val_loss=1.2122 val_acc=0.6636 lr=0.000835
[05/15] train_loss=0.6859 train_acc=0.7907 val_loss=1.0870 val_acc=0.6962 lr=0.000750
[06/15] train_loss=0.5074 train_acc=0.8398 val_loss=1.0306 val_acc=0.7226 lr=0.000655
[07/15] train_loss=0.3915 train_acc=0.8759 val_loss=0.9914 val_acc=0.7396 lr=0.000552
[08/15] train_loss=0.2672 train_acc=0.9146 val_loss=0.9648 val_acc=0.7520 lr=0.000448
[09/15] train_loss=0.1771 train_acc=0.9450 val_loss=0.9487 val_acc=0.7642 lr=0.000345
[10/15] train_loss=0.1115 train_acc=0.9664 val_loss=0.9254 val_acc=0.7700 lr=0.000250
[11/15] train_loss=0.0694 train_acc=0.9812 val_loss=0.9162 val_acc=0.7764 lr=0.000165
[12/15] train_loss=0.0440 train_acc=0.9897 val_loss=0.

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



===== Final Result =====
best_val_acc : 0.7908 (epoch 15)
test_loss    : 0.8994
test_acc     : 0.7934


epoch,▁▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇██
lr,██▇▇▆▆▅▄▃▃▂▂▁▁▁
test/acc,▁
test/loss,▁
train/acc,▁▃▄▅▅▆▆▇▇██████
train/loss,█▅▄▄▃▃▂▂▂▁▁▁▁▁▁
val/acc,▁▄▅▅▆▆▇▇▇██████
val/loss,█▅▄▃▃▂▂▂▂▁▁▁▁▁▁
best_epoch,15
best_val_acc,0.7908
epoch,15


{'best_val_acc': 0.7908,
 'best_epoch': 15,
 'test_loss': 0.8994423023223876,
 'test_acc': 0.7934}

In [ ]:
#augmentation 강화
config_aug = dict(BASELINE_CONFIG)
config_aug["run_name"] = "resnet18_adamw_lr1e-3_bs128_strongaug"
config_aug["augmentation"] = "strong"

run_experiment(config_aug)

100%|██████████| 169M/169M [00:03<00:00, 42.7MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 196MB/s]


[01/15] train_loss=1.9109 train_acc=0.4841 val_loss=1.6478 val_acc=0.5414 lr=0.000989
[02/15] train_loss=1.2158 train_acc=0.6449 val_loss=1.2623 val_acc=0.6340 lr=0.000957
[03/15] train_loss=0.9572 train_acc=0.7128 val_loss=1.1481 val_acc=0.6760 lr=0.000905
[04/15] train_loss=0.7786 train_acc=0.7623 val_loss=1.0618 val_acc=0.6934 lr=0.000835
[05/15] train_loss=0.6365 train_acc=0.8034 val_loss=1.0140 val_acc=0.7098 lr=0.000750
[06/15] train_loss=0.4922 train_acc=0.8450 val_loss=0.9814 val_acc=0.7242 lr=0.000655
[07/15] train_loss=0.3670 train_acc=0.8837 val_loss=0.9197 val_acc=0.7450 lr=0.000552
[08/15] train_loss=0.2564 train_acc=0.9198 val_loss=0.8748 val_acc=0.7650 lr=0.000448
[09/15] train_loss=0.1796 train_acc=0.9443 val_loss=0.8787 val_acc=0.7676 lr=0.000345
[10/15] train_loss=0.1168 train_acc=0.9666 val_loss=0.8441 val_acc=0.7826 lr=0.000250
[11/15] train_loss=0.0743 train_acc=0.9797 val_loss=0.8235 val_acc=0.7882 lr=0.000165
[12/15] train_loss=0.0499 train_acc=0.9885 val_loss=0.

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



===== Final Result =====
best_val_acc : 0.8046 (epoch 15)
test_loss    : 0.8076
test_acc     : 0.7981


epoch,▁▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇██
lr,██▇▇▆▆▅▄▃▃▂▂▁▁▁
test/acc,▁
test/loss,▁
train/acc,▁▃▄▅▅▆▆▇▇██████
train/loss,█▅▄▄▃▃▂▂▂▁▁▁▁▁▁
val/acc,▁▃▅▅▅▆▆▇▇▇█████
val/loss,█▅▄▃▃▃▂▂▂▁▁▁▁▁▁
best_epoch,15
best_val_acc,0.8046
epoch,15


{'best_val_acc': 0.8046,
 'best_epoch': 15,
 'test_loss': 0.8075677680969239,
 'test_acc': 0.7981}

In [ ]:
#backbone 변경
backbone_cfg = dict(BASELINE_CONFIG)
backbone_cfg["run_name"] = "efficientnetb0_adamw_lr1e-3_bs64_basicaug"
backbone_cfg["model_name"] = "efficientnet_b0"
backbone_cfg["batch_size"] = 64

run_experiment(backbone_cfg)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 161MB/s]


[01/15] train_loss=1.4397 train_acc=0.6040 val_loss=0.9645 val_acc=0.7154 lr=0.000989
[02/15] train_loss=0.8427 train_acc=0.7467 val_loss=0.8451 val_acc=0.7564 lr=0.000957
[03/15] train_loss=0.6442 train_acc=0.8042 val_loss=0.7974 val_acc=0.7656 lr=0.000905
[04/15] train_loss=0.5077 train_acc=0.8406 val_loss=0.8320 val_acc=0.7764 lr=0.000835
[05/15] train_loss=0.3997 train_acc=0.8734 val_loss=0.7416 val_acc=0.7908 lr=0.000750
[06/15] train_loss=0.3037 train_acc=0.9017 val_loss=0.7540 val_acc=0.7986 lr=0.000655
[07/15] train_loss=0.2143 train_acc=0.9288 val_loss=0.7471 val_acc=0.7998 lr=0.000552
[08/15] train_loss=0.1550 train_acc=0.9489 val_loss=0.7608 val_acc=0.8134 lr=0.000448
[09/15] train_loss=0.1067 train_acc=0.9648 val_loss=0.7338 val_acc=0.8258 lr=0.000345
[10/15] train_loss=0.0684 train_acc=0.9783 val_loss=0.7344 val_acc=0.8272 lr=0.000250
[11/15] train_loss=0.0428 train_acc=0.9875 val_loss=0.7503 val_acc=0.8340 lr=0.000165
[12/15] train_loss=0.0315 train_acc=0.9911 val_loss=0.

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



===== Final Result =====
best_val_acc : 0.8456 (epoch 14)
test_loss    : 0.7306
test_acc     : 0.8463


epoch,▁▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇██
lr,██▇▇▆▆▅▄▃▃▂▂▁▁▁
test/acc,▁
test/loss,▁
train/acc,▁▄▅▅▆▆▇▇▇██████
train/loss,█▅▄▃▃▂▂▂▁▁▁▁▁▁▁
val/acc,▁▃▄▄▅▅▆▆▇▇▇▇███
val/loss,█▅▃▄▂▂▂▂▂▂▂▁▁▁▁
best_epoch,14
best_val_acc,0.8456
epoch,15


{'best_val_acc': 0.8456,
 'best_epoch': 14,
 'test_loss': 0.7306084965705871,
 'test_acc': 0.8463}

In [ ]:
#장기학습 1
final_cfg = dict(BASELINE_CONFIG)
final_cfg["run_name"] = "FINAL_efficientnetb0_adamw_lr1e-3_bs64_basicaug_ep30"
final_cfg["model_name"] = "efficientnet_b0"
final_cfg["batch_size"] = 64
final_cfg["learning_rate"] = 1e-3
final_cfg["optimizer"] = "AdamW"
final_cfg["augmentation"] = "basic"
final_cfg["epochs"] = 30
final_cfg["log_pred_every"] = 10

final_cfg["num_workers"] = 0
final_cfg["pin_memory"] = True

import gc
gc.collect()
torch.cuda.empty_cache()

run_experiment(final_cfg)


In [17]:
test_cfg = dict(final_cfg)
test_cfg["run_name"] = "TEST_efficientnetb0_debug_ep2"
test_cfg["epochs"] = 2
test_cfg["log_pred_every"] = 9999
test_cfg["num_workers"] = 0

test_result = run_experiment(test_cfg)
test_result

c:\Users\dochi\Desktop\wandb_hw\venv313\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


[01/2] train_loss=1.4439 train_acc=0.6024 val_loss=0.8916 val_acc=0.7320 lr=0.000500
[02/2] train_loss=0.6387 train_acc=0.8058 val_loss=0.6500 val_acc=0.8008 lr=0.000000


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.



===== Final Result =====
best_val_acc : 0.8008 (epoch 2)
test_loss    : 0.6581
test_acc     : 0.8037


epoch,▁▁██
lr,█▁
test/acc,▁
test/loss,▁
train/acc,▁█
train/loss,█▁
val/acc,▁█
val/loss,█▁
best_epoch,2
best_val_acc,0.8008
epoch,2


{'best_val_acc': 0.8008,
 'best_epoch': 2,
 'test_loss': 0.6581069147586822,
 'test_acc': 0.8037}

In [ ]:
#장기학습 2
final_cfg_2 = dict(config_aug)
final_cfg_2["run_name"] = "FINAL_resnet18_adamw_lr1e-3_bs128_strongaug_ep30"
final_cfg_2["epochs"] = 30
final_cfg_2["log_pred_every"] = 10

run_experiment(final_cfg_2)